# Topic 1: Overview of Generative AI

## The Problem We Are Solving

Barclays processes thousands of customer complaints every day - fraud disputes, fee complaints,
transfer failures, account access issues. Today a human reads each one and decides:
- What category is this?
- How urgent is it?
- Which team handles it?

Your job as an engineering team: build an AI system that automates this triage.

## What You Will Learn Today

By the end of this notebook you will be able to:
1. Name the four major families of generative AI models and their trade-offs
2. Explain in one sentence why each family is (or is not) suited for text understanding
3. Call the OpenAI API with a system prompt, a user message, tokens, and temperature
4. Write a complaint triage prompt that classifies and routes a Barclays complaint

## How This Notebook Is Organised

- Section 1: The generative AI landscape - a map of the territory
- Section 2: Why the wrong model family fails (live demo)
- Section 3: Autoregressive LLMs - the right tool for text
- Section 4: LLM-as-a-Service - the OpenAI API in practice

In [ ]:
# Environment setup for Topic 1
# All code runs in the SageMaker Studio notebook kernel (no remote training jobs)

!pip install -q "numpy<2" "openai>=1.0.0"

import numpy as np
import random
import json
import getpass

# PyTorch for the GAN demo in Section 2
import torch
import torch.nn as nn
import torch.optim as optim

print("numpy version :", np.__version__)
print("torch version  :", torch.__version__)
print("Setup complete.")

## Section 1: The Generative AI Landscape

Before we pick a tool, we need a map.

**Discriminative AI** learns a boundary: given input X, predict label Y.
A spam classifier is discriminative. A fraud detector is discriminative.

**Generative AI** learns the distribution of data itself: it can *create* new X values
that look like they came from the training set. A model that writes new complaint text,
draws a new image, or synthesises a new audio clip - that is generative.

There are four main families. They all generate new data, but they do it in completely
different ways. Understanding *how* each works is what will let you choose the right one.

### The Four Families at a Glance

| Family | Core idea | Best at | Weakness |
|--------|-----------|---------|----------|
| GAN (Generative Adversarial Network) | Generator vs Discriminator game | Photorealistic images | Unstable training, mode collapse, bad for discrete text |
| VAE (Variational Autoencoder) | Encode to latent space, sample, decode | Smooth interpolation, anomaly detection | Blurry images, limited quality ceiling |
| Diffusion | Iterative denoising from pure noise | State-of-the-art image and audio quality | Very slow inference (hundreds of steps) |
| Autoregressive | Predict next token given all previous tokens | Text, code, structured sequences | Sequential - cannot parallelise generation |

The rest of Section 1 unpacks each row in turn.

The table above is your map for this session. Three of the four families dominate *image*
generation. Autoregressive is the family that dominates *text*. That asymmetry is not an
accident - you will see exactly why in Section 2 when we try (and fail) to use a GAN for text.

In [ ]:
# ------------------------------------------------------------------
# BEAT 1: A minimal GAN - works for numbers, breaks for text
# ------------------------------------------------------------------
# A GAN has two networks:
#   Generator (G): takes random noise -> produces fake data
#   Discriminator (D): takes data -> outputs P(real)
# They play a game. G tries to fool D; D tries to catch G.

# --- Part A: GAN working correctly on continuous data (numbers) ---

torch.manual_seed(42)

class TinyGenerator(nn.Module):
    def __init__(self, noise_dim=8, output_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 16), nn.ReLU(),
            nn.Linear(16, output_dim)
        )
    def forward(self, z):
        return self.net(z)

class TinyDiscriminator(nn.Module):
    def __init__(self, input_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

G = TinyGenerator()
D = TinyDiscriminator()
opt_G = optim.Adam(G.parameters(), lr=0.01)
opt_D = optim.Adam(D.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

# Train for 200 steps on "real data" = samples from N(5, 1)
for step in range(200):
    real = torch.randn(32, 1) + 5.0        # real data: mean=5
    noise = torch.randn(32, 8)
    fake = G(noise)

    # Discriminator step
    opt_D.zero_grad()
    d_real = D(real)
    d_fake = D(fake.detach())
    loss_D = loss_fn(d_real, torch.ones(32, 1)) + loss_fn(d_fake, torch.zeros(32, 1))
    loss_D.backward()
    opt_D.step()

    # Generator step
    opt_G.zero_grad()
    loss_G = loss_fn(D(G(noise)), torch.ones(32, 1))
    loss_G.backward()
    opt_G.step()

# Sample from the trained generator
noise = torch.randn(10, 8)
samples = G(noise).detach().numpy().flatten()
print("GAN on numbers - generated values (should be near 5.0):")
print([round(float(s), 2) for s in samples])
print()

# --- Part B: Now try to generate TEXT with the same approach ---
# Imagine our vocabulary is just 5 words:
vocab = ["complaint", "fraud", "account", "payment", "urgent"]
vocab_size = len(vocab)

# A naive GAN for text would output a one-hot vector over vocab
# and we pick argmax to get a "word". Let's try:
class TextGenerator(nn.Module):
    def __init__(self, noise_dim=8, vocab_size=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 16), nn.ReLU(),
            nn.Linear(16, vocab_size)
        )
    def forward(self, z):
        return torch.softmax(self.net(z), dim=-1)  # distribution over vocab

text_G = TextGenerator()
noise = torch.randn(1, 8)
output = text_G(noise)
word_index = output.argmax(dim=-1).item()
print("Naive GAN text output (before training):", vocab[word_index])
print()
print("--- THE PROBLEM ---")
print("To backpropagate through the discriminator into the generator,")
print("we need gradients to flow from D's output back through the selected word.")
print("But argmax is NOT differentiable. You cannot nudge 'fraud' toward 'fraud + 0.001'.")
print("There is no such thing as a word between two words.")
print()
print("Result: standard GAN training completely breaks for discrete text.")
print("(SeqGAN and similar work around this with reinforcement learning,")
print(" but that introduces its own instability problems.)")

<!-- DIAGRAM: The autoregressive LLM token generation loop. Shows: (1) context window containing prompt tokens, (2) arrow to softmax distribution over vocabulary, (3) sampling a token (affected by temperature), (4) appending the new token back to context window, (5) repeat arrow. A second branch shows how temperature=0 picks the peak token (greedy) vs temperature=1.0 spreads the distribution (creative). -->

```mermaid
graph TD
    prompt["Context window\n(prompt tokens so far)"] --> forward["Full forward pass\nthrough transformer"]
    forward --> dist["Softmax distribution\nover vocabulary (~100k tokens)"]
    dist --> temp{"Temperature"}
    temp -->|"temp=0 (greedy)"| peak["Pick peak token\n(most likely)"]
    temp -->|"temp=1.0 (creative)"| sample["Sample from\nspread distribution"]
    peak --> append["Append new token\nto context window"]
    sample --> append
    append --> stop{"End of sequence\ntoken?"}
    stop -->|"No"| prompt
    stop -->|"Yes"| output["Final generated text"]

    style prompt fill:#ddf,stroke:#33c
    style dist fill:#ffd,stroke:#aa3
    style output fill:#dfd,stroke:#3a3
```

*Figure: The four model families each generate output differently. GANs and diffusion models output continuous tensors in one or many steps. Autoregressive models output one discrete token at a time, looping until the sequence is complete. This token-by-token loop is what makes autoregressive the natural fit for text, where each word depends on every word before it.*

### Why GANs Fail for Text (The Gradient Problem)

The generator needs a gradient signal: "was this output good or bad, and in which direction
should I change?" For continuous data (pixel values, audio samples) that direction is a real
number. For discrete tokens the direction does not exist - words are not on a number line.

This is not a solvable engineering problem. It is a mathematical mismatch between the GAN
training recipe and the nature of language.

**So where do GANs shine?** Photorealistic face synthesis (StyleGAN), image-to-image
translation (Pix2Pix), and any task where the output is a continuous tensor.

**Peer Discussion (3 min)**

Think about a Barclays fraud detection system:
- Would you use a GAN to *classify* whether a complaint is about fraud?
- Would you use a GAN to *generate synthetic* complaint text for a test dataset?
- What are the risks of using synthetic complaint text in financial services?

Discuss with the person next to you. There is no single right answer.

### VAEs: Smooth Latent Space, Blurry Outputs

A Variational Autoencoder encodes input into a *probability distribution* in latent space
(not a single point), then samples from that distribution to decode a new output.

**The upside**: the latent space is smooth and continuous. Interpolating between two points
gives meaningful intermediate outputs. Great for anomaly detection: a complaint that lands
far from all training clusters is likely unusual.

**The downside for text**: same gradient problem as GANs when the output is discrete tokens.
VAEs are used for text *embeddings* (encoding), not text *generation*.

### Diffusion Models: SOTA Images, Wrong Tool for Text

Diffusion models learn to reverse a noise-adding process. Training: take an image, add
Gaussian noise step by step until it is pure noise. Learn to predict and remove that noise.
Inference: start from noise, denoise for 50-1000 steps.

**The upside**: state-of-the-art image quality (Stable Diffusion, DALL-E image backbone,
Midjourney). Also used for audio (MusicGen) and molecule design.

**The downside for text**: you cannot add Gaussian noise to a discrete token index.
Some research applies diffusion to continuous embeddings (MDLM, PLAID) but these are
not production-ready and are orders of magnitude slower than autoregressive LLMs.

In [ ]:
# VAE in 15 lines: encode -> sample from latent distribution -> decode
# This is the CONCEPT, not a trained model. Just to make it concrete.

torch.manual_seed(0)

# Encoder: maps input to (mu, log_var) - a Gaussian in latent space
def encode(x, W_mu, W_logvar):
    mu = torch.matmul(x, W_mu)
    log_var = torch.matmul(x, W_logvar)
    return mu, log_var

# Reparameterisation trick: sample z = mu + eps * std (DIFFERENTIABLE)
def reparameterise(mu, log_var):
    std = torch.exp(0.5 * log_var)
    eps = torch.randn_like(std)   # random noise
    return mu + eps * std         # shift and scale

# Simulate encoding a complaint embedding (dim=4) to latent dim=2
x = torch.tensor([[0.8, 0.1, 0.5, 0.2]])  # a "complaint" vector
W_mu    = torch.randn(4, 2)
W_logvar = torch.randn(4, 2)

mu, log_var = encode(x, W_mu, W_logvar)
z1 = reparameterise(mu, log_var)
z2 = reparameterise(mu, log_var)  # second sample from same input

print("Same complaint input -> two different latent samples:")
print("  z1:", z1.detach().numpy().round(3))
print("  z2:", z2.detach().numpy().round(3))
print()
print("This is the VAE trade-off: stochastic latent space enables generation")
print("but the decoder must reconstruct from a blurry, noisy latent - hence blurry outputs.")

## Section 2: The Right Tool - Autoregressive Models

So far:
- GANs: gradient problem with discrete tokens
- VAEs: good for embeddings, not text generation
- Diffusion: SOTA for images, research-only for text

All three were designed around *continuous data*. Text is *discrete*.

### The Autoregressive Insight

What if we did not try to generate the whole complaint at once?
What if we generated it one word (actually: one token) at a time,
conditioning each new token on everything we have produced so far?

This is the autoregressive idea:

  P(token_1, token_2, ..., token_n) = P(token_1) * P(token_2 | token_1) * ...

No gradient problem: we train with cross-entropy on each token prediction.
Standard backpropagation. Scales to billions of parameters.

This is what GPT-4o, Claude, Llama, and every modern LLM is doing.

### The Autoregressive Loop in Detail

The diagram above shows the generation loop for any autoregressive model, from the tiny
GRU you just saw to GPT-4o. Every step is a full forward pass through the model. The only
inputs are the tokens produced so far; the only output is a probability distribution over
the next token.

Temperature controls how peaked that distribution is. At temperature 0 the model always
picks the single most probable token. At temperature 2 it spreads probability mass across
many tokens, producing more varied (and more error-prone) output.

You will implement this loop from scratch in the code cell below.

In [ ]:
# ------------------------------------------------------------------
# BEAT 3: Autoregressive generation loop - made explicit
# ------------------------------------------------------------------
# We build the world's smallest autoregressive language model:
#   - Vocabulary: digits 0-9 plus a few characters
#   - Architecture: embedding -> 1-layer GRU -> linear -> softmax
#   - Training: 150 steps on a trivial sequence ("01234567890123456789")
# The goal is to SEE the loop, not to impress anyone with quality.

torch.manual_seed(7)

# --- Vocabulary ---
chars = list("0123456789")
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

# --- Tiny model ---
embed_dim = 8
hidden_dim = 16

embedding = nn.Embedding(vocab_size, embed_dim)
gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
linear = nn.Linear(hidden_dim, vocab_size)

params = list(embedding.parameters()) + list(gru.parameters()) + list(linear.parameters())
optimizer = optim.Adam(params, lr=0.05)
loss_fn = nn.CrossEntropyLoss()

# --- Training data: predict next digit in a repeating sequence ---
sequence = "01234567890123456789"
input_ids  = torch.tensor([[char_to_idx[c] for c in sequence[:-1]]])  # shape (1, 19)
target_ids = torch.tensor([[char_to_idx[c] for c in sequence[1:]]])   # shifted by 1

print("Training tiny autoregressive model...")
for step in range(150):
    optimizer.zero_grad()
    emb = embedding(input_ids)          # (1, 19, 8)
    out, _ = gru(emb)                   # (1, 19, 16)
    logits = linear(out)                # (1, 19, vocab_size)
    loss = loss_fn(logits.view(-1, vocab_size), target_ids.view(-1))
    loss.backward()
    optimizer.step()

print(f"Final loss: {loss.item():.4f}")
print()

# --- The generation loop ---
# This is the EXACT same loop that GPT-4o uses at inference time.
# The only difference is scale (billions of parameters vs ~2K here).

def generate(seed_char, n_tokens=10, temperature=1.0):
    result = [seed_char]
    # Start with the seed token
    current_idx = torch.tensor([[char_to_idx[seed_char]]])
    hidden = None

    for _ in range(n_tokens):
        # Step 1: embed the current token
        emb = embedding(current_idx)            # (1, 1, 8)
        # Step 2: pass through model, update hidden state
        out, hidden = gru(emb, hidden)          # (1, 1, 16)
        logits = linear(out[:, -1, :])          # (1, vocab_size)

        # Step 3: apply temperature (lower = more deterministic)
        logits = logits / temperature
        probs = torch.softmax(logits, dim=-1)

        # Step 4: sample the next token from the distribution
        next_idx = torch.multinomial(probs, num_samples=1)  # (1, 1)

        # Step 5: append to result, use as next input
        result.append(idx_to_char[next_idx.item()])
        current_idx = next_idx.unsqueeze(0)     # shape for next step

    return "".join(result)

print("Autoregressive generation from seed '0':")
print("  temperature=0.1 (greedy)  :", generate("0", n_tokens=10, temperature=0.1))
print("  temperature=1.0 (balanced):", generate("0", n_tokens=10, temperature=1.0))
print("  temperature=2.0 (random)  :", generate("0", n_tokens=10, temperature=2.0))
print()
print("Notice: lower temperature = more predictable (repeats the learned pattern).")
print("        higher temperature = more random (deviates from the pattern).")

### Lab 1: Temperature and Autoregressive Generation (Tier 1 - Guided)

**Situation**: You are evaluating whether an autoregressive model is suitable for generating
standardised complaint response templates at Barclays. You need to understand how temperature
controls the trade-off between consistency and variation.

**Task**: Run the generate() function with four different temperature values and record
what happens to the output.

**Action**: Fill in the temperatures in the starter code below, run the cell, and
answer the discussion questions in the markdown cell that follows.

**Result**: After completing the lab, you should be able to explain in one sentence
what temperature does and when you would use a low vs high value in production.

**Stretch (fast finishers)**: Modify generate() to use greedy decoding (argmax instead of
multinomial sampling) and compare it to temperature=0.1. Are they identical? Why or why not?

**Homework Extension**: See the cell after the safety-net cell.

In [ ]:
# Lab 1: Autoregressive Temperature Exploration - SOLUTION
# We pick four temperatures that span the useful range and clearly show the effect.

# Step 1: choose four temperature values to explore (must be > 0)
# Good choices cover: near-deterministic, balanced, and creative/random.
# 0.1 = almost greedy (very consistent), 0.5 = slightly varied,
# 1.0 = the default in most APIs (balanced), 2.0 = highly random
temperatures_to_test = [0.1, 0.5, 1.0, 2.0]

# --- do not edit below this line ---
if temperatures_to_test is not None:
    print("Seed character: '5'")
    print("-" * 40)
    for t in temperatures_to_test:
        output = generate("5", n_tokens=12, temperature=t)
        print(f"  temperature={t:<4} -> {output}")
    print()
    print("Observation: at very low temperature, the model is deterministic/repetitive.")
    print("             at very high temperature, the model is random/unpredictable.")

In [ ]:
# Lab 1 safety-net: run this if you did not finish Lab 1.
# SKIP this cell if you DID finish Lab 1.
if temperatures_to_test is None:
    print("Using Lab 1 safety-net so the rest of the notebook can run.")
    temperatures_to_test = [0.1, 0.5, 1.0, 2.0]

**Homework Extension - Lab 1**

The tiny model you used was trained on a repeating digit sequence.
Real LLMs are trained on trillions of tokens of text.

1. Read the OpenAI API documentation on the `temperature` parameter:
   https://platform.openai.com/docs/api-reference/chat/create

2. What is the default temperature for GPT-4o? What range is supported?

3. For each Barclays use case below, decide whether you would use a low, medium,
   or high temperature value, and justify your choice in 1-2 sentences:
   a. Generating a standardised rejection letter for a loan application
   b. Generating creative marketing copy for a new current account product
   c. Classifying an incoming complaint into one of 10 predefined categories

Write your answers in a markdown cell in a new notebook and bring them to the next session.

## Section 3: LLM-as-a-Service - The OpenAI API

We have established:
- GAN, VAE, Diffusion: wrong tools for text understanding and generation
- Autoregressive LLMs: the right tool, and you now know why at the code level

**The conclusion for our complaint system**:
We need an autoregressive LLM that understands text, and we need it *right now* -
not after training a model from scratch on Barclays data for six months.

This is where LLM-as-a-Service comes in.

### The Major Players (2025-2026)

| Provider | Model | Strengths |
|----------|-------|-----------|
| OpenAI | GPT-4o, GPT-4o-mini | Best general reasoning, huge ecosystem |
| Anthropic | Claude Sonnet/Opus | Strong reasoning, large context, safety focus |
| Google | Gemini Pro/Ultra | Multimodal, long context |
| Meta | Llama 3.x (open weights) | Self-hostable, fine-tunable |
| Mistral | Mistral Large | European, GDPR-friendly, open weights |

For this course we use **GPT-4o via the OpenAI API**.
All the concepts (tokens, temperature, system prompts, fine-tuning) apply equally to
every provider - only the API surface changes.

### Three Concepts You Must Understand Before Calling Any LLM API

1. **Tokens**: the atomic unit of text. Not words, not characters - subword pieces.
   "Barclays" is 1 token. "uncharacteristically" might be 4-6 tokens.
   You pay per token. You are rate-limited per token.

2. **Temperature**: you just built this from scratch. Range 0-2 in GPT-4o.
   Lower = deterministic. Higher = creative (and sometimes wrong).

3. **System prompt**: a privileged message that sets the model's role and rules.
   It appears before the user's message and is invisible to the end user.
   This is where you put: "You are a Barclays complaint triage agent. ..."

In [ ]:
# Securely enter your OpenAI API key.
# This uses getpass() so the key is never visible in the notebook output
# and never stored in the notebook file (do not hardcode it).

openai_api_key = getpass.getpass("Paste your OpenAI API key and press Enter: ")

from openai import OpenAI
client = OpenAI(api_key=openai_api_key)

print("OpenAI client initialised. Key ends in:", openai_api_key[-4:])

### Beat 1: Calling the API Without a System Prompt

What happens if we just send the complaint text with no instructions?

In [ ]:
# Beat 1: Call GPT-4o with no system prompt - just the raw complaint text.
# Watch what happens.

complaint_text = (
    "I tried to make a payment of 500 pounds to my landlord last night and it just "
    "disappeared. The money left my account but my landlord says he never received it. "
    "The app just shows 'pending' with no further information. I need this resolved "
    "urgently as my rent is now overdue and I am being threatened with eviction."
)

# No system prompt - just a user message
response_no_system = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": complaint_text}
    ],
    temperature=0.7,
    max_tokens=200
)

print("Response WITHOUT a system prompt:")
print("-" * 50)
print(response_no_system.choices[0].message.content)
print("-" * 50)
print()
print("Tokens used:")
print("  prompt_tokens    :", response_no_system.usage.prompt_tokens)
print("  completion_tokens:", response_no_system.usage.completion_tokens)
print("  total_tokens     :", response_no_system.usage.total_tokens)
print()
print("Notice: the response is empathetic but does not classify, route, or follow any format.")
print("This is useless for an automated triage system.")

### Beat 2: The System Prompt - Setting the Role and Rules

The model's response without a system prompt is empathetic but useless for operations:
it gives advice but does not classify, does not route, and does not follow any format.

<!-- DIAGRAM: openai-api-request-response-flow -->

```mermaid
sequenceDiagram
    participant nb as Notebook (client)
    participant api as OpenAI API
    participant model as gpt-4o model

    nb->>api: HTTP POST /chat/completions\n{system_prompt, user_message, temperature}
    api->>model: Route request to model instance
    model->>model: Apply system prompt constraints
    model->>model: Sample tokens\n(temperature shapes distribution)
    model->>api: Completion tokens
    api->>nb: JSON response\n{choices[0].message.content}
```

*Figure: The diagram shows the full request/response cycle: the system prompt and user message are combined into a single HTTP request; the model returns a completion shaped by temperature (low = peaked distribution, high = spread distribution) and the system prompt constraints.*

A system prompt lets us give the model:
1. A **role** (who it is)
2. **Rules** (what it must and must not do)
3. An **output format** (JSON, structured text, etc.)
4. **Context** (what categories exist, what routing means)

The model will then apply these constraints to every user message it receives.
This is the difference between a general assistant and a production triage agent.

**Token note**: the system prompt counts toward your total token usage on every call.
A well-crafted, concise system prompt costs less than a verbose one - and often works better.

In [ ]:
# ------------------------------------------------------------------
# BEAT 3: Full working triage call with a system prompt
# ------------------------------------------------------------------

# A good system prompt has four components:
#   1. Role: who you are
#   2. Rules: what you must/must-not do
#   3. Categories: the routing options available
#   4. Output format: exactly what to return

system_prompt = """You are a complaint triage agent for Barclays Bank.
Your job is to read an incoming customer complaint and produce a structured triage decision.

Rules:
- You must respond in JSON only. No prose before or after the JSON.
- Do not include your reasoning in the output - only the structured decision.
- If the complaint is unclear, set category to 'unclassified'.

Categories (pick exactly one):
- 'payment_failure'  : money sent but not received, payment stuck or pending
- 'fraud_dispute'    : suspected unauthorised transaction or account takeover
- 'account_access'   : cannot log in, locked out, password or 2FA issues
- 'fee_complaint'    : unexpected charge, disputed fee
- 'unclassified'     : does not fit any category above

Output format (JSON only):
{
  "category": "<one of the five categories>",
  "urgency": "high | medium | low",
  "summary": "<one sentence, max 20 words>",
  "routing_team": "<Payments | Fraud | Digital | Fees | Triage>"
}"""

# The complaint (same one as Beat 1)
complaint_text = (
    "I tried to make a payment of 500 pounds to my landlord last night and it just "
    "disappeared. The money left my account but my landlord says he never received it. "
    "The app just shows 'pending' with no further information. I need this resolved "
    "urgently as my rent is now overdue and I am being threatened with eviction."
)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": system_prompt},  # sets the role and rules
        {"role": "user",   "content": complaint_text}  # the complaint
    ],
    temperature=0,       # 0 = deterministic - we want consistent triage decisions
    max_tokens=200
)

# Extract the text content
raw_output = response.choices[0].message.content
print("Raw API response:")
print(raw_output)
print()

# Parse the JSON
try:
    triage = json.loads(raw_output)
    print("Parsed triage decision:")
    print(f"  Category     : {triage['category']}")
    print(f"  Urgency      : {triage['urgency']}")
    print(f"  Summary      : {triage['summary']}")
    print(f"  Routing team : {triage['routing_team']}")
except json.JSONDecodeError as e:
    print("JSON parse error:", e)
    print("Raw output was:", raw_output)

print()
print("Token usage:")
print(f"  prompt_tokens    : {response.usage.prompt_tokens}")
print(f"  completion_tokens: {response.usage.completion_tokens}")
print(f"  total_tokens     : {response.usage.total_tokens}")
print()
print("At $0.005 per 1K output tokens (gpt-4o as of 2025),")
print(f"this call cost approximately ${response.usage.total_tokens / 1000 * 0.005:.5f}")

### Lab 2: Build a Barclays Complaint Triage Prompt (Tier 2 - Hard)

**Situation**: The Barclays digital team has signed off on a complaint routing system
powered by GPT-4o. You are the engineer responsible for the system prompt.
The triage logic must handle five complaint categories and produce a JSON response
that the downstream routing system can parse without ambiguity.

**Task**: Write a system prompt that correctly classifies all five test complaints
below. Your prompt must:
1. Define the model's role clearly
2. List the five categories with unambiguous descriptions
3. Specify the exact JSON output format (category, urgency, summary, routing_team)
4. Handle the edge case: a complaint that fits two categories (pick the primary one)

**Action**: Fill in `my_system_prompt` below and run the test harness.
The test harness sends all five complaints and prints the triage decisions.
You have succeeded when all five complaints produce valid, sensible JSON.

**Result**: A working system prompt that classifies five real-world-style complaints.

**Stretch**: Add a sixth field `"confidence": "high | medium | low"` to your output format
and adjust the system prompt so the model sets confidence=low when the complaint is ambiguous.

In [ ]:
# Lab 2: Build a Barclays Complaint Triage Prompt - SOLUTION
# A well-structured system prompt has: role, rules, categories, output format.

# Step 1: Write your system prompt
# The key insight: define categories with unambiguous trigger phrases, and
# add an explicit tie-breaking rule for ambiguous cases (complaint 5).
my_system_prompt = """You are a complaint triage agent for Barclays Bank.
Your job is to read an incoming customer complaint and produce a structured triage decision.

Rules:
- You must respond in JSON only. No prose before or after the JSON.
- Do not include your reasoning - only the structured decision.
- If the complaint is unclear, set category to 'unclassified'.
- When a complaint could fit two categories, choose the one with the greater financial risk.

Categories (pick exactly one):
- 'payment_failure'  : money sent but not received, payment stuck, pending, or missing
- 'fraud_dispute'    : unauthorised transaction, unknown recipient, suspected account takeover
- 'account_access'   : cannot log in, locked out, MFA or phone number issues
- 'fee_complaint'    : unexpected or disputed charge, overdraft fee, monthly fee
- 'unclassified'     : does not fit any category above

Urgency rules:
- high   : financial loss occurring now, or account is completely blocked
- medium : issue is inconvenient but no immediate financial loss
- low    : informational query or minor complaint

Routing teams:
- Payments : for payment_failure
- Fraud    : for fraud_dispute
- Digital  : for account_access
- Fees     : for fee_complaint
- Triage   : for unclassified

Output format (JSON only, no markdown, no prose):
{
  "category": "<one of the five categories>",
  "urgency": "high | medium | low",
  "summary": "<one sentence, max 20 words>",
  "routing_team": "<Payments | Fraud | Digital | Fees | Triage>"
}"""

# --- Test complaints (do not edit) ---
test_complaints = [
    # Complaint 1: payment failure
    "I sent 1200 pounds to my sister three days ago and it still has not arrived. "
    "The transaction shows as completed on my end but she has nothing.",

    # Complaint 2: fraud dispute
    "I just got an alert for a 350 pound transaction at a shop in Manchester. "
    "I have not been to Manchester and I did not make this purchase. "
    "Please block my card immediately.",

    # Complaint 3: account access
    "I cannot log in to the app. It keeps asking for a code sent to my old phone number "
    "but I changed my number six months ago and never updated it.",

    # Complaint 4: fee complaint
    "You charged me a 25 pound overdraft fee last month but my account was only overdrawn "
    "for four hours overnight. This seems completely disproportionate.",

    # Complaint 5: ambiguous - could be payment_failure OR fraud
    # Solution: the 'greater financial risk' rule routes this to fraud_dispute
    # because money went to an unknown, unauthorised recipient.
    "300 pounds left my account last night to someone called 'M. Johnson' but I have never "
    "heard of this person and I did not authorise any transfer to them.",
]

# --- Test harness (do not edit) ---
if my_system_prompt is not None:
    print("Running triage on 5 complaints...")
    print("=" * 60)
    for i, complaint in enumerate(test_complaints, 1):
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": my_system_prompt},
                {"role": "user",   "content": complaint}
            ],
            temperature=0,
            max_tokens=150
        )
        raw = resp.choices[0].message.content
        print(f"Complaint {i}:")
        try:
            t = json.loads(raw)
            print(f"  category : {t.get('category', 'MISSING')}")
            print(f"  urgency  : {t.get('urgency', 'MISSING')}")
            print(f"  summary  : {t.get('summary', 'MISSING')}")
        except json.JSONDecodeError:
            print("  [JSON parse failed] raw output:", raw[:100])
        print()
else:
    print("my_system_prompt is None - fill it in and re-run this cell.")

In [ ]:
# Lab 2 safety-net: run this if you did not finish Lab 2.
# SKIP this cell if you DID finish Lab 2.
if my_system_prompt is None:
    print("Using Lab 2 safety-net so the rest of the notebook can run.")
    my_system_prompt = """You are a complaint triage agent for Barclays Bank.
Respond in JSON only with fields: category, urgency, summary, routing_team.
Categories: payment_failure, fraud_dispute, account_access, fee_complaint, unclassified.
Urgency: high (financial loss or account access blocked), medium, low.
Routing teams: Payments, Fraud, Digital, Fees, Triage."""

**Homework Extension - Lab 2**

Your triage system is working in the notebook. Now extend it:

1. **Few-shot prompting**: Add two worked examples to your system prompt
   (a "user" message with a complaint and an "assistant" message with the correct JSON).
   Does this change the quality of the output on complaint 5 (the ambiguous one)?

2. **Edge cases**: Test your prompt on:
   - A complaint written entirely in capital letters (angry customer)
   - A complaint that is not about banking at all ("I want to complain about the weather")
   - A one-word complaint: "Fraud!"
   Does your prompt handle these gracefully? What changes would you make?

3. **Cost estimation**: Calculate the daily API cost if Barclays processes 10,000 complaints
   per day, using the token counts from your test harness calls.
   Assume the average complaint is the same length as complaint 1.
   Is this cost acceptable? What levers do you have to reduce it?

Write your answers and revised prompts in a new notebook section.

**Peer Discussion (4 min)**

You have just built a working complaint triage system using the OpenAI API.
Before Barclays could deploy this to production, what questions must the engineering
team answer? Consider:

1. **Reliability**: What happens when the OpenAI API is down? Does the complaint sit
   in a queue, or does a human take over? How do you detect the failure?

2. **Consistency**: We set temperature=0. But GPT-4o can still change between API
   versions (OpenAI updates the model). How do you test for regression?

3. **Compliance**: A Barclays complaint is personal financial data (PII). What are
   the data residency and regulatory implications of sending it to OpenAI's servers?
   (Think: GDPR, FCA, PCI-DSS.)

4. **Cost at scale**: If triage volume spikes 10x during an outage (when many customers
   complain simultaneously), what happens to your API costs and rate limits?

These are not rhetorical questions - the engineering team at Barclays is dealing with
exactly these trade-offs today. You will revisit them in the deployment topic later in the course.

In [ ]:
# Practical: count tokens before you make the API call
# This avoids surprises on the bill and helps you stay within rate limits.
#
# We use tiktoken, OpenAI's tokenizer library.
# Note: you do not need an API key to count tokens.

try:
    import tiktoken
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tiktoken"])
    import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")

# Count tokens in our system prompt and a complaint
system_tokens = enc.encode(my_system_prompt)
complaint_tokens = enc.encode(test_complaints[0])

print("Token counts:")
print(f"  system prompt  : {len(system_tokens)} tokens")
print(f"  complaint 1    : {len(complaint_tokens)} tokens")
print(f"  total input    : {len(system_tokens) + len(complaint_tokens)} tokens")
print()

# Show tokenisation of a tricky word
word = "Barclays"
tokens = enc.encode(word)
print(f"How '{word}' is tokenised:")
print(f"  token ids: {tokens}")
print(f"  tokens   : {[enc.decode([t]) for t in tokens]}")
print()

word2 = "uncharacteristically"
tokens2 = enc.encode(word2)
print(f"How '{word2}' is tokenised:")
print(f"  token ids: {tokens2}")
print(f"  tokens   : {[enc.decode([t]) for t in tokens2]}")
print()
print("Rule of thumb: 1 token ~ 0.75 English words. 100 tokens ~ 75 words.")

### Preview: Few-Shot Prompting

Your Lab 2 system prompt is a **zero-shot** prompt: you describe what you want but give no examples.
GPT-4o can also learn from **examples in the prompt** - this is called few-shot prompting.

In the messages array, you can add pairs of ("user": complaint, "assistant": correct JSON)
before the actual complaint. The model treats these as demonstrations.

```python
messages = [
    {"role": "system",    "content": my_system_prompt},
    # Few-shot example 1
    {"role": "user",      "content": "I never authorised this 50 pound charge."},
    {"role": "assistant", "content": '{"category": "fraud_dispute", "urgency": "high", '
                                     '"summary": "Unauthorised 50 pound charge disputed.", '
                                     '"routing_team": "Fraud"}'},
    # Few-shot example 2
    {"role": "user",      "content": "Why was I charged a monthly fee?"},
    {"role": "assistant", "content": '{"category": "fee_complaint", "urgency": "low", '
                                     '"summary": "Customer queries monthly account fee.", '
                                     '"routing_team": "Fees"}'},
    # The actual complaint
    {"role": "user",      "content": complaint_text}
]
```

Each example pair costs tokens. More examples = higher cost = (usually) better quality.
Finding the right number is a prompt engineering decision you will practise in Topic 12.

The Homework Extension for Lab 2 asks you to try this - bring your results to the next session.

In [ ]:
# Final demonstration: run triage on three complaints and display a routing table.
# This is the core of the Barclays complaint intelligence system you will build
# throughout this course.

demo_complaints = [
    ("Payment stuck for 3 days to landlord, eviction threatened.", "Expected: payment_failure / high"),
    ("Unauthorised 200 pound transaction appeared this morning.",   "Expected: fraud_dispute / high"),
    ("Cannot log in, my old phone number is on the account.",      "Expected: account_access / medium"),
]

print("Barclays Complaint Triage System - Demo Run")
print("=" * 65)
print(f"{'#':<3} {'Category':<20} {'Urgency':<8} {'Team':<10} {'Note'}")
print("-" * 65)

for i, (complaint, note) in enumerate(demo_complaints, 1):
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": my_system_prompt},
            {"role": "user",   "content": complaint}
        ],
        temperature=0,
        max_tokens=100
    )
    try:
        t = json.loads(resp.choices[0].message.content)
        cat  = t.get("category",      "???")[:18]
        urg  = t.get("urgency",       "???")[:6]
        team = t.get("routing_team",  "???")[:8]
        print(f"{i:<3} {cat:<20} {urg:<8} {team:<10} {note}")
    except json.JSONDecodeError:
        print(f"{i:<3} [JSON error]")

print("=" * 65)
print()
print("System works. This is the foundation we build on for the rest of the course.")

## Wrap-Up: What You Built Today

In the last 50 minutes you:

1. **Mapped the generative AI landscape**: GAN, VAE, Diffusion, Autoregressive.
   You understand the key trade-offs and why three of the four families are not
   the right tool for text understanding.

2. **Saw the gradient problem**: You ran code that shows exactly why GANs cannot
   generate text - the argmax over a discrete vocabulary is not differentiable.

3. **Built the autoregressive loop from scratch**: embedding -> GRU -> softmax ->
   sample -> append -> repeat. You know this is the exact loop GPT-4o runs,
   just at a different scale.

4. **Called the OpenAI API**: system prompt, user message, temperature, token counts.
   You have a working complaint triage prototype for Barclays.

### Key Numbers to Remember

- Temperature range for GPT-4o: 0 to 2 (0 = deterministic, 2 = very random)
- Rule of thumb: 1 token is approximately 0.75 English words
- Zero-shot (no examples) vs few-shot (examples in prompt): more examples = more tokens = more cost
- argmax is not differentiable: this is why GANs fail for text

### What Comes Next (Topic 2: Introducing LLMs)

Topic 2 goes inside the black box. You will see:
- What a transformer architecture actually looks like in code
- What attention is and why it solves the long-range dependency problem
- How training works: cross-entropy loss, next-token prediction at scale

The complaint triage system continues - in Topic 2 you will understand *why* GPT-4o
is so much better at understanding "eviction threatened" (context) than older models.

### Questions to Bring to Topic 2

- Why does a longer context window cost more? (Hint: think about the attention mechanism.)
- If we remove the system prompt, does the model still produce JSON? Try it as homework.
- What would it take to fine-tune GPT-4o on Barclays complaint data? (You will do this in Topic 6.)

In [ ]:
# Notebook complete. Summary of what was created in this session.

print("Topic 1 Complete - Session Summary")
print("=" * 45)
print(f"  OpenAI client    : initialised (key ends ...{openai_api_key[-4:]})")
print(f"  System prompt    : {len(my_system_prompt)} characters, "
      f"{len(enc.encode(my_system_prompt))} tokens")
print(f"  Temperatures tested: {temperatures_to_test}")
print()
print("Variables available for Topic 2:")
print("  client           : OpenAI client (if key is still valid)")
print("  my_system_prompt : your triage prompt from Lab 2")
print("  test_complaints  : list of 5 test complaints")
print()
print("Next: Topic 2 - Introducing LLMs (what is inside the black box?)")